# 02 - Tratamento dos Dados

- Origem: arquivos csv coletados no notebook '01_coleta.ipynb'

- Objetivo: Converter tipos, verificar nulos e consolidar SELIC, IPCA E câmbio em um único DataFrame

In [1]:
import pandas as pd

In [2]:
selic = pd.read_csv('../data/raw/selic.csv')
ipca = pd.read_csv('../data/raw/ipca.csv')
cambio = pd.read_csv('../data/raw/cambio.csv')

In [17]:
def exploracao_inicial(df):
    print(df.head())
    print(df.dtypes)
    
    print("\nShape: ", df.shape)
    print("\nDados nulos: \n", df.isnull().sum())

In [18]:
exploracao_inicial(selic)

         data  valor
0  01/01/2020    4.5
1  02/01/2020    4.5
2  03/01/2020    4.5
3  04/01/2020    4.5
4  05/01/2020    4.5
data      object
valor    float64
dtype: object

Shape:  (2368, 2)

Dados nulos: 
 data     0
valor    0
dtype: int64


In [19]:
exploracao_inicial(ipca)

         data  valor
0  01/01/2020   0.21
1  01/02/2020   0.25
2  01/03/2020   0.07
3  01/04/2020  -0.31
4  01/05/2020  -0.38
data      object
valor    float64
dtype: object

Shape:  (77, 2)

Dados nulos: 
 data     0
valor    0
dtype: int64


In [20]:
exploracao_inicial(cambio)

         data   valor
0  02/01/2020  4.0213
1  03/01/2020  4.0522
2  06/01/2020  4.0554
3  07/01/2020  4.0841
4  08/01/2020  4.0672
data      object
valor    float64
dtype: object

Shape:  (1626, 2)

Dados nulos: 
 data     0
valor    0
dtype: int64


In [31]:
selic['data'] = pd.to_datetime(selic['data'], format="%d/%m/%Y")
selic['valor'] = pd.to_numeric(selic['valor'], errors='coerce')

ipca['data'] = pd.to_datetime(ipca['data'], format="%d/%m/%Y")
ipca['valor'] = pd.to_numeric(ipca['valor'], errors='coerce')

cambio['data'] = pd.to_datetime(cambio['data'], format="%d/%m/%Y")
cambio['valor'] = pd.to_numeric(cambio['valor'], errors='coerce')

In [33]:
print(selic.dtypes)
print(ipca.dtypes)
print(cambio.dtypes)

data     datetime64[ns]
valor           float64
dtype: object
data     datetime64[ns]
valor           float64
dtype: object
data     datetime64[ns]
valor           float64
dtype: object


In [40]:
selic_mensal = selic.resample('ME', on='data')['valor'].last()
ipca_mensal = ipca.set_index('data')['valor']
ipca_mensal.index = ipca_mensal.index + pd.offsets.MonthEnd(0)
cambio_mensal = cambio.resample('ME', on='data')['valor'].mean()

In [41]:
df_macro = pd.concat([selic_mensal, ipca_mensal, cambio_mensal], axis=1)
df_macro.columns = ['selic', 'ipca', 'cambio']
print(df_macro.head())
print(df_macro.shape)

            selic  ipca    cambio
data                             
2020-01-31   4.50  0.21  4.149464
2020-02-29   4.25  0.25  4.341011
2020-03-31   3.75  0.07  4.883855
2020-04-30   3.75 -0.31  5.325580
2020-05-31   3.00 -0.38  5.643445
(78, 3)


In [42]:
df_macro.to_csv('../data/processed/macro_mensal.csv')

- Shape: 78 linhas x 3 colunas (selic, ipca, cambio)
- Período Jan/2020 - mai/2026
- Nulos: Nennhum
- Agregação mensal: SELIC com .last() e câmbio com .mean() 
- Alinhamento de índice do IPCA com MonthEnd(0)